In [1]:
import random
import numpy as np
from PIL import Image

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.v2 import Compose, Normalize

# Custom imports
%run -i ../fix_path.py
from data_generation.image_classification import generate_dataset
from stepbystep.v0 import StepByStep

## Convolution
**What a convolution is?**

A convolution is an operation where a small matrix (called a filter or kernel) slides over an image.
At each position, it multiplies its values with the pixels underneath (receptive field) and sums them into one output value.
Repeating this across the entire image produces a new transformed image called a feature map.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5/chapter_5-img1.png" style="width:50%;">
  <img src="../../images/chapter_5/chapter_5-img2.png" style="width:50%;">
</div>

In [ ]:
# batch_size=1, n_channels=1, image_size=6x6
single = np.array([[
        [
            [5, 0, 8, 7, 8, 1],
            [1, 9, 5, 0, 7, 7],
            [6, 0, 2, 4, 6, 6],
            [9, 7, 6, 6, 8, 4],
            [8, 3, 8, 5, 1, 3],
            [7, 2, 7, 0, 1, 0]
        ]
    ]])
single.shape

(1, 1, 6, 6)

In [ ]:
# batch_size=1, n_channels=1, kernel_size=3x3
identity = np.array([[
    [
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]
    ]
]])

identity.shape

(1, 1, 3, 3)

In [4]:
region = single[:, :, 0:3, 0:3]
filtered_region = region * identity
filtered_region, filtered_region.sum()

(array([[[[0, 0, 0],
          [0, 9, 0],
          [0, 0, 0]]]]),
 np.int64(9))

## Filters/kernels

A filter (or kernel) is a small grid of numbers (like 3×3 or 5×5) that acts as a pattern detector.
Different filters detect different visual patterns, such as:

- edges (horizontal, vertical, diagonal)
- corners
- textures
- smooth regions
- color transitions

Filters start with random numbers and learn useful patterns during training through backpropagation.
<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5/chapter_5-img3.png" style="width:50%;">
  <img src="../../images/chapter_5/chapter_5-img4.png" style="width:50%;">
</div>

## What convolutions achieve

Convolutions allow a neural network to:

- extract local spatial patterns (edges, shapes, textures)
- build hierarchical understanding (edges → shapes → objects)
- preserve spatial structure (which pixels are near each other)
- reduce parameters dramatically (same filter is reused everywhere)
- detect patterns regardless of where they appear in the image

The result is a set of feature maps that describe what important visual features are present and where.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5/chapter_5-img5.png" style="width:50%;">
  <img src="../../images/chapter_5/chapter_5-img6.png" style="width:50%;">
</div>

## Why we need them

We need convolutions because:

- Images have spatial structure, and treating pixels independently (like flattening) destroys that structure.
- Convolutions are efficient: far fewer parameters than fully connected layers.
- They provide translation invariance — a feature detected in one corner is recognized anywhere.
- They enable deep networks to learn visual concepts from simple local patterns to complex objects.

## How to Determine the Size of the Feature Map given an Input Image
**Without Padding (Valid Convolution)**
$$
\begin{align*}
H_{\text{out}} &= H_{\text{in}} - K_h + 1 \\
W_{\text{out}} &= W_{\text{in}} - K_w + 1
\end{align*}
$$

**With padding $P$ (same convolution or padded convolution)**
$$
\begin{align*}
H_{\text{out}} &= \left( \frac{H_{\text{in}} + 2P - K_h}{S} \right) + 1 \\
W_{\text{out}} &= \left( \frac{W_{\text{in}} + 2P - K_w}{S} \right) + 1
\end{align*}
$$
**Where:**
- $H_{out}$, $W_{out}$ = size of the feature map
- $H_{in}$, $W_{in}$ = input image height and width
- $K_h$, $K_w$ = kernel height and width
- $P$ = padding
- $S$ = stride

**PS: The larger the filter, the smaller the resulting image.**

## Convolving in PyTorch

In [5]:
image = torch.as_tensor(single).float()
kernel_identity = torch.as_tensor(identity).float()
image, kernel_identity

(tensor([[[[5., 0., 8., 7., 8., 1.],
           [1., 9., 5., 0., 7., 7.],
           [6., 0., 2., 4., 6., 6.],
           [9., 7., 6., 6., 8., 4.],
           [8., 3., 8., 5., 1., 3.],
           [7., 2., 7., 0., 1., 0.]]]]),
 tensor([[[[0., 0., 0.],
           [0., 1., 0.],
           [0., 0., 0.]]]]))

Just like the activation functions we saw in Chapter 4, convolutions come in two flavors: functional and module. There is a fundamental difference between the two, though: The functional convolution takes the kernel / filter as an argument while the module has (learnable) weights to represent the kernel / filter.

In [6]:
convolved = F.conv2d(image, kernel_identity, stride=1)
convolved

tensor([[[[9., 5., 0., 7.],
          [0., 2., 4., 6.],
          [7., 6., 6., 8.],
          [3., 8., 5., 1.]]]])

Now, let’s turn our attention to PyTorch’s convolution module, nn.Conv2d. It has many arguments; let’s focus on the first four of them:
- `in_channels`: number of channels of the input image
- `out_channels`: number of channels produced by the convolution
- `kernel_size`: size of the (square) convolution filter / kernel
- `stride`: the size of the movement of the selected region

There are a couple of things to notice here. First, there is no argument for the kernel / filter itself, there is only a kernel_size argument.

The actual filter, that is, the square matrix used to perform element-wise multiplication, is learned by the module.

Second, it is possible to produce multiple channels as output. It simply means the module is going to learn multiple filters. Each filter is going to produce a different
result, which is being called a channel here.
So far, we’ve been using a single-channel image as input, and applying one filter (size three by three) to it, moving one pixel at a time, resulting in one output per
channel. Let’s do it in code:

In [7]:
conv = nn.Conv2d(
in_channels=1, out_channels=1, kernel_size=3, stride=1
)
conv(image)

tensor([[[[ 0.9031, -2.6643, -0.3971, -3.9930],
          [-0.3868, -0.9480,  0.4866, -1.7961],
          [-0.1389, -1.7837,  0.5113, -1.3513],
          [-0.0884, -2.0899,  1.1207, -1.0201]]]],
       grad_fn=<ConvolutionBackward0>)

## Stride Size 
The number of pixels to move the filter as we apply it on the image. So far we have been using a stride of 1, now we'll try out a stride of size 2.

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5/chapter_5-img7.png" style="width:50%;">
  <img src="../../images/chapter_5/chapter_5-img8.png" style="width:50%;">
</div>


In [8]:
convolved_stride2 = F.conv2d(image, kernel_identity, stride=2)
convolved_stride2

tensor([[[[9., 0.],
          [7., 6.]]]])

## Padding
Notice that applying a filter to an image shrinks it (as can be seen in the result of the feature map). What if we wanted to maintain the shape of the original image? To do that, we use padding where we sorround the image with zeros effectively increasing it's size and maintaining it's shape after convolution.

See? By adding columns and rows of zeros around it, we expand the input image such that the gray region starts centered in the actual top left corner of the input image. This simple trick can be used to preserve the original size of the image.

![chapter_5-img9](../../images/chapter_5/chapter_5-img9.png)

In code, as usual, PyTorch gives us two options: functional (F.pad()) and module (nn.ConstantPad2d). Let’s start with the module version this time:

In [9]:
constant_padder = nn.ConstantPad2d(padding=1, value=0)
constant_padder(image)

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

There are two arguments: padding, for the number of columns and rows to be stuffed in the image; and value, for the value that is filling these new columns and rows. One can also do asymmetric padding by specifying a tuple in the padding argument representing (left, right, top, bottom). So, if we were to stuff our image on the left and right sides only, the argument would go like this: (1, 1, 0, 0). 

**In the functional version, one must specify the padding as a tuple.**

We can achieve the same result using the functional padding:

In [10]:
padded = F.pad(image, pad=(1, 1, 1, 1), mode='constant', value=0)
padded

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

## Types of Padding

### 1. Constant Padding
In constant padding, a constant value is used to fill the extra padded rows and columns

![chapter_5-img10](../../images/chapter_5/chapter_5-img10.png)

In [13]:
# Modular Approach
constant_padder = nn.ConstantPad2d(padding=1, value=0)
constant_padder(image)

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

In [15]:
# Functional Approach
F.pad(input=image, pad=(1, 1, 1, 1), value=0, mode="constant")

tensor([[[[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 5., 0., 8., 7., 8., 1., 0.],
          [0., 1., 9., 5., 0., 7., 7., 0.],
          [0., 6., 0., 2., 4., 6., 6., 0.],
          [0., 9., 7., 6., 6., 8., 4., 0.],
          [0., 8., 3., 8., 5., 1., 3., 0.],
          [0., 7., 2., 7., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]]]])

### Replication Padding
In replication padding, the padded pixels have the same value as the closest real pixel. The padded corners have the same value as the real corners. The other columns (left and right) and rows (top and bottom) replicate the corresponding values of the original image. The values used in the replication are in a darker shade of orange.

![chapter_5-img11](../../images/chapter_5/chapter_5-img11.png)

In [16]:
# Modular Approach
replication_padder = nn.ReplicationPad2d(padding=1)
replication_padder(image)

tensor([[[[5., 5., 0., 8., 7., 8., 1., 1.],
          [5., 5., 0., 8., 7., 8., 1., 1.],
          [1., 1., 9., 5., 0., 7., 7., 7.],
          [6., 6., 0., 2., 4., 6., 6., 6.],
          [9., 9., 7., 6., 6., 8., 4., 4.],
          [8., 8., 3., 8., 5., 1., 3., 3.],
          [7., 7., 2., 7., 0., 1., 0., 0.],
          [7., 7., 2., 7., 0., 1., 0., 0.]]]])

In [19]:
# Functional Approach
F.pad(input=image, pad=(1, 1, 1, 1), mode="replicate")

tensor([[[[5., 5., 0., 8., 7., 8., 1., 1.],
          [5., 5., 0., 8., 7., 8., 1., 1.],
          [1., 1., 9., 5., 0., 7., 7., 7.],
          [6., 6., 0., 2., 4., 6., 6., 6.],
          [9., 9., 7., 6., 6., 8., 4., 4.],
          [8., 8., 3., 8., 5., 1., 3., 3.],
          [7., 7., 2., 7., 0., 1., 0., 0.],
          [7., 7., 2., 7., 0., 1., 0., 0.]]]])

### Reflection Padding
In reflection padding, the outermost columns and rows of the original image act as the axes of reflection. A good way of understanding how reflection padding works is imagining the outermost columns and rows are mirrors. So, the left padded column (forget about the corners for now) reflects the second column (since the first column is the axis of reflection). The same reasoning goes for the right padded column. Similarly, the top padded row reflects the second row (since the first row is the axis of reflection), and the same reasoning goes for the bottom padded row.

![chapter_5-img12](../../images/chapter_5/chapter_5-img12.png)

The values used in the reflection are in a darker shade of orange. The corners have the same values as the intersection of the reflected rows and columns of the original image. Hopefully, the image can convey the idea better.

In [20]:
# Modular Approach
reflection_padder = nn.ReflectionPad2d(padding=1)
reflection_padder(image)

tensor([[[[9., 1., 9., 5., 0., 7., 7., 7.],
          [0., 5., 0., 8., 7., 8., 1., 8.],
          [9., 1., 9., 5., 0., 7., 7., 7.],
          [0., 6., 0., 2., 4., 6., 6., 6.],
          [7., 9., 7., 6., 6., 8., 4., 8.],
          [3., 8., 3., 8., 5., 1., 3., 1.],
          [2., 7., 2., 7., 0., 1., 0., 1.],
          [3., 8., 3., 8., 5., 1., 3., 1.]]]])

In [21]:
# Functional Approach
F.pad(input=image, pad=(1, 1, 1, 1), mode="reflect")

tensor([[[[9., 1., 9., 5., 0., 7., 7., 7.],
          [0., 5., 0., 8., 7., 8., 1., 8.],
          [9., 1., 9., 5., 0., 7., 7., 7.],
          [0., 6., 0., 2., 4., 6., 6., 6.],
          [7., 9., 7., 6., 6., 8., 4., 8.],
          [3., 8., 3., 8., 5., 1., 3., 1.],
          [2., 7., 2., 7., 0., 1., 0., 1.],
          [3., 8., 3., 8., 5., 1., 3., 1.]]]])

### Circular Padding
Circular padding is like replication padding only that the replicate goes to the opposite of the actual (original image) row or column. For example, consider the leftmost column in the original image ([5, 1, 6, 9, 8, 7]), this values get coppied to the rightmost ede of the padded image. Similarly, the rightmost column in the original image ([1, 7, 6, 4, 3, 0]) gets copied to the leftmost column in the padded image. The same logic applies for corners.

![chapter_5-img13](../../images/chapter_5/chapter_5-img13.png)

In [23]:
# Modular Approach
circular_padder = nn.CircularPad2d(padding=1)
circular_padder(image)

tensor([[[[0., 7., 2., 7., 0., 1., 0., 7.],
          [1., 5., 0., 8., 7., 8., 1., 5.],
          [7., 1., 9., 5., 0., 7., 7., 1.],
          [6., 6., 0., 2., 4., 6., 6., 6.],
          [4., 9., 7., 6., 6., 8., 4., 9.],
          [3., 8., 3., 8., 5., 1., 3., 8.],
          [0., 7., 2., 7., 0., 1., 0., 7.],
          [1., 5., 0., 8., 7., 8., 1., 5.]]]])

In [24]:
# Functional Approach
F.pad(input=image, pad=(1, 1, 1, 1), mode="circular")

tensor([[[[0., 7., 2., 7., 0., 1., 0., 7.],
          [1., 5., 0., 8., 7., 8., 1., 5.],
          [7., 1., 9., 5., 0., 7., 7., 1.],
          [6., 6., 0., 2., 4., 6., 6., 6.],
          [4., 9., 7., 6., 6., 8., 4., 9.],
          [3., 8., 3., 8., 5., 1., 3., 8.],
          [0., 7., 2., 7., 0., 1., 0., 7.],
          [1., 5., 0., 8., 7., 8., 1., 5.]]]])

## Edge Detector Filter

In [25]:
edge = np.array([[
    [
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0]
    ]
]])

edge.shape

(1, 1, 3, 3)

In [26]:
kernel_edge = torch.as_tensor(edge).float()
kernel_edge.shape

torch.Size([1, 1, 3, 3])

### Applying the edge detector kernel to the padded image

<div style="display:flex; gap:10px;">
  <img src="../../images/chapter_5/chapter_5-img14.png" style="width:50%;">
  <img src="../../images/chapter_5/chapter_5-img15.png" style="width:50%;">
</div>

In [ ]:
conv_padded =F.conv2d(padded, kernel_edge, stride=1)
conv_padded

tensor([[[[-19.,  22., -20., -12., -17.,  11.],
          [ 16., -30.,  -1.,  23.,  -7., -14.],
          [-14.,  24.,   7.,  -2.,   1.,  -7.],
          [-15., -10.,  -1.,  -1., -15.,   1.],
          [-13.,  13., -11.,  -5.,  13.,  -7.],
          [-18.,   9., -18.,  13.,  -3.,   4.]]]])

## Pooling
Pooling is an operation used in Convolutional Neural Networks (CNNs) to reduce the spatial size (height and width) of feature maps.
It works by sliding a small window (e.g., 2×2 or 3×3) across the feature map and computing a summary statistic inside that window.

**What does pooling do / why do we need it?**

✔ **1. Downsamples feature maps:** It reduces resolution (e.g., from 28×28 to 14×14), which decreases computation and reduces memory usage

✔ **2. Makes features more robust to small translations:** Pooling makes the network less sensitive to small shifts or distortions in the input. 

✔ **3. Helps prevent overfitting:** By reducing spatial detail and parameter count.

✔ **4. Keeps important features:** Especially max pooling keeps the strongest activation in each region.